# QMI2D validation — $D_s^+\to\pi^-\pi^+\pi^+$

This notebook demonstrates the direct two-dimensional extension of QMI. Every Dalitz cell carries one complex value $A_{ij}=a_{ij}e^{i\phi_{ij}}$. The same field is evaluated with `none`, `linear`, and `cubic` interpolation. For the two identical $\pi^+$ particles we use folded coordinates $(s_{low},s_{high})=(\min(s_{12},s_{13}),\max(s_{12},s_{13}))$.

In [ ]:
import numpy as np
import jax
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DalitzAmplitude, DecayChannel, DecayModel, QMI2D, RealImag,
    enable_x64, weighted_resample,
)
enable_x64()

channel = DecayChannel("D_s+", ("pi-", "pi+", "pi+"))
smin = (channel.daughter_masses[0] + channel.daughter_masses[1])**2
smax = (channel.parent_mass - channel.daughter_masses[2])**2
edges = np.linspace(smin, smax, 9)
centers = 0.5 * (edges[:-1] + edges[1:])
xx, yy = np.meshgrid(centers, centers, indexing="ij")

# Illustrative truth field on bin centers. Only the folded half is physically used.
magnitudes = 1.0 + 0.8*np.exp(-((xx-0.9)**2 + (yy-1.8)**2)/0.35) + 0.25*np.sin(1.5*xx)
phases = 0.4 + 1.1*xx - 0.55*yy + 0.35*np.sin(2.0*yy)

def make_field(mode):
    return QMI2D(
        s12_edges=tuple(edges), s13_edges=tuple(edges),
        magnitudes=tuple(tuple(float(v) for v in row) for row in magnitudes),
        phases=tuple(tuple(float(v) for v in row) for row in phases),
        interpolation=mode, folded=True,
    )


## Compare interpolation modes on the physical Dalitz region

In [ ]:
base_model = DecayModel(
    channel,
    [DalitzAmplitude("qmi2d", make_field("none"), RealImag(1.0, 0.0))],
    normalization_resolution=180,
)
grid = base_model.normalization_sample
data = grid.as_dict()

fig, axes = plt.subplots(3, 2, figsize=(12, 15), constrained_layout=True)
for row, mode in enumerate(("none", "linear", "cubic")):
    field = make_field(mode)
    mag, phase = field.interpolated_magnitude_phase(data)
    sc0 = axes[row,0].scatter(np.asarray(grid.s12), np.asarray(grid.s13), c=np.asarray(mag), s=3)
    fig.colorbar(sc0, ax=axes[row,0], label="magnitude")
    axes[row,0].set(title=f"{mode}: magnitude", xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
    sc1 = axes[row,1].scatter(np.asarray(grid.s12), np.asarray(grid.s13), c=np.asarray(phase), s=3)
    fig.colorbar(sc1, ax=axes[row,1], label="phase [rad]")
    axes[row,1].set(title=f"{mode}: phase", xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]")
plt.show()

## Use the cubic field as a coherent Dalitz amplitude and generate a toy

In [ ]:
truth_field = make_field("cubic")
model = DecayModel(
    channel,
    [DalitzAmplitude("qmi2d", truth_field, RealImag(1.0, 0.0))],
    normalization_resolution=350,
)

norm = model.normalization_sample
intensity = np.asarray(model.intensity(norm.as_dict()))
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(norm.s12), np.asarray(norm.s13), bins=120, weights=np.asarray(norm.weights)*intensity)
fig.colorbar(h[3], ax=ax, label=r"grid weight $\times |A|^2$")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]", title="QMI2D cubic intensity")
plt.show()

N_POOL, N_TOY = 300_000, 50_000
pool = model.generate_phase_space(N_POOL, seed=6110)
target = pool.weights * model.intensity(pool.as_dict())
toy = weighted_resample(jax.random.key(6111), pool, target, N_TOY, replace=True)
fig, ax = plt.subplots(figsize=(7,6))
h = ax.hist2d(np.asarray(toy.s12), np.asarray(toy.s13), bins=100)
fig.colorbar(h[3], ax=ax, label="events")
ax.set(xlabel=r"$s_{12}$ [GeV$^2$]", ylabel=r"$s_{13}$ [GeV$^2$]", title=r"$D_s^+\to\pi^-\pi^+\pi^+$ QMI2D toy")
plt.show()

## Fit parameters

To fit the field, replace any entries of `magnitudes` and `phases` by `Parameter.dynamics(...)` objects owned by the `DalitzAmplitude`. One global magnitude/phase convention should be fixed. In a realistic 2D fit we should start with a coarse grid and inspect correlations/Hessian eigenvalues before increasing the number of cells.